# Segregation Simple Extension 3 — NetLogo para Python/Mesa

Este notebook implementa em **Python + Mesa** uma adaptação do modelo **Segregation Simple Extension 3** do NetLogo.

O modelo representa agentes de diferentes etnias/cores em uma grade. Cada agente avalia sua vizinhança e fica feliz apenas se atender a duas condições:

1. Ter uma porcentagem mínima de vizinhos da mesma etnia.
2. Ter uma porcentagem mínima de vizinhos de etnias diferentes.

Agentes infelizes se movem para novas posições livres, gerando padrões espaciais de segregação ao longo do tempo.

Referência do modelo original:

Wilensky, U., Rand, W. (2006). *NetLogo Segregation Simple Extension 3 model*. Center for Connected Learning and Computer-Based Modeling, Northwestern University.


## 1. Instalação das dependências

Execute a célula abaixo se o ambiente ainda não tiver `mesa`, `pandas` e `matplotlib` instalados.


In [ ]:
# Se necessário, descomente e execute:
# !pip install mesa pandas matplotlib


## 2. Importações


In [ ]:
import random
import pandas as pd
import matplotlib.pyplot as plt

from mesa import Agent, Model
from mesa.space import SingleGrid
from mesa.datacollection import DataCollector


## 3. Classe do agente

Esta classe representa cada *turtle* do NetLogo.

Cada agente possui:

- etnia/cor;
- estado de felicidade;
- número de vizinhos similares;
- número de vizinhos diferentes;
- total de vizinhos;
- limiar individual de similaridade desejada.


In [ ]:
class SegregationAgent(Agent):
    """
    Agente equivalente à turtle do NetLogo.
    """

    def __init__(self, model, ethnicity, color, similar_wanted):
        super().__init__(model)

        self.ethnicity = ethnicity
        self.color = color

        # Equivalente a my-%-similar-wanted
        self.my_percent_similar_wanted = similar_wanted

        self.happy = False
        self.similar_nearby = 0
        self.other_nearby = 0
        self.total_nearby = 0

    def update_happiness(self):
        """
        Equivalente ao procedimento update-turtles do NetLogo.
        Avalia os 8 patches vizinhos ao redor da posição atual.
        """

        neighbors = self.model.grid.get_neighbors(
            self.pos,
            moore=True,
            include_center=False
        )

        self.total_nearby = len(neighbors)

        self.similar_nearby = sum(
            1 for agent in neighbors
            if agent.ethnicity == self.ethnicity
        )

        self.other_nearby = sum(
            1 for agent in neighbors
            if agent.ethnicity != self.ethnicity
        )

        if self.total_nearby == 0:
            # No NetLogo, se total-nearby = 0,
            # as duas condições são satisfeitas porque os requisitos viram 0.
            self.happy = True
            return

        similar_required = (
            self.my_percent_similar_wanted * self.total_nearby / 100
        )

        different_required = (
            self.model.percent_different_wanted * self.total_nearby / 100
        )

        self.happy = (
            self.similar_nearby >= similar_required
            and self.other_nearby >= different_required
        )

    def find_new_spot(self):
        """
        Adaptação do find-new-spot do NetLogo.

        No NetLogo:
        rt random-float 360
        fd random-float 10
        if any? other turtles-here [ find-new-spot ]
        setxy pxcor pycor

        Aqui o agente procura uma célula vazia em um raio local.
        Se não encontrar, escolhe qualquer célula vazia do grid.
        """

        x, y = self.pos
        width = self.model.grid.width
        height = self.model.grid.height
        radius = self.model.move_radius

        candidate_positions = []

        for dx in range(-radius, radius + 1):
            for dy in range(-radius, radius + 1):
                if dx == 0 and dy == 0:
                    continue

                nx = (x + dx) % width
                ny = (y + dy) % height

                if self.model.grid.is_cell_empty((nx, ny)):
                    candidate_positions.append((nx, ny))

        if candidate_positions:
            new_pos = self.random.choice(candidate_positions)
            self.model.grid.move_agent(self, new_pos)
            return

        empty_cells = list(self.model.grid.empties)

        if empty_cells:
            new_pos = self.random.choice(empty_cells)
            self.model.grid.move_agent(self, new_pos)

    def step(self):
        """
        O agente só se move se estiver infeliz.
        """

        if not self.happy:
            self.find_new_spot()


## 4. Classe do modelo

Esta classe representa o mundo da simulação.

Ela contém:

- grade espacial;
- criação dos agentes;
- atualização das condições de felicidade;
- movimentação dos agentes infelizes;
- cálculo das métricas globais.


In [ ]:
class SegregationModel(Model):
    """
    Modelo Mesa equivalente ao Segregation Simple Extension 3 do NetLogo.
    """

    def __init__(
        self,
        width=50,
        height=50,
        number=2000,
        number_of_ethnicities=5,
        percent_similar_wanted=30,
        percent_different_wanted=5,
        move_radius=10,
        seed=None
    ):
        super().__init__(seed=seed)

        self.width = width
        self.height = height
        self.number = number
        self.number_of_ethnicities = number_of_ethnicities
        self.percent_similar_wanted = percent_similar_wanted
        self.percent_different_wanted = percent_different_wanted
        self.move_radius = move_radius

        self.tick = 0
        self.running = True

        self.colors = [
            "red",
            "green",
            "yellow",
            "blue",
            "orange"
        ]

        if self.number_of_ethnicities > len(self.colors):
            raise ValueError(
                f"O modelo tem apenas {len(self.colors)} cores definidas. "
                f"Reduza number_of_ethnicities ou adicione mais cores."
            )

        max_agents = width * height

        if number > max_agents:
            raise ValueError(
                f"number={number} é maior que o número de células do grid "
                f"({max_agents})."
            )

        self.grid = SingleGrid(width, height, torus=True)

        self.datacollector = DataCollector(
            model_reporters={
                "tick": lambda m: m.tick,
                "percent_similar": lambda m: m.percent_similar,
                "percent_unhappy": lambda m: m.percent_unhappy,
                "num_happy": lambda m: m.num_happy,
                "num_unhappy": lambda m: m.num_unhappy,
            },
            agent_reporters={
                "ethnicity": "ethnicity",
                "color": "color",
                "happy": "happy",
                "similar_nearby": "similar_nearby",
                "other_nearby": "other_nearby",
                "total_nearby": "total_nearby",
                "my_percent_similar_wanted": "my_percent_similar_wanted",
            }
        )

        self.percent_similar = 0
        self.percent_unhappy = 0
        self.num_happy = 0
        self.num_unhappy = 0

        self.setup_agents()
        self.update_turtles()
        self.update_globals()
        self.datacollector.collect(self)

    def setup_agents(self):
        """
        Equivalente ao setup do NetLogo.

        Cria NUMBER agentes em patches aleatórios.
        Cada agente recebe uma cor/etnia aleatória entre as etnias disponíveis.
        Cada agente recebe um limiar individual aleatório de similaridade:
        random %-similar-wanted.
        """

        all_positions = [
            (x, y)
            for x in range(self.width)
            for y in range(self.height)
        ]

        selected_positions = self.random.sample(all_positions, self.number)

        for pos in selected_positions:
            ethnicity = self.random.randrange(self.number_of_ethnicities)
            color = self.colors[ethnicity]

            # Equivalente a:
            # set my-%-similar-wanted random %-similar-wanted
            similar_wanted = self.random.randrange(
                max(1, self.percent_similar_wanted + 1)
            )

            agent = SegregationAgent(
                model=self,
                ethnicity=ethnicity,
                color=color,
                similar_wanted=similar_wanted
            )

            self.grid.place_agent(agent, pos)

    def update_turtles(self):
        """
        Equivalente ao update-turtles do NetLogo.
        """

        for agent in list(self.agents):
            agent.update_happiness()

    def update_globals(self):
        """
        Equivalente ao update-globals do NetLogo.
        """

        agents = list(self.agents)

        similar_neighbors = sum(a.similar_nearby for a in agents)
        total_neighbors = sum(a.total_nearby for a in agents)

        if total_neighbors == 0:
            self.percent_similar = 0
        else:
            self.percent_similar = (
                similar_neighbors / total_neighbors
            ) * 100

        self.num_unhappy = sum(1 for a in agents if not a.happy)
        self.num_happy = len(agents) - self.num_unhappy

        if len(agents) == 0:
            self.percent_unhappy = 0
        else:
            self.percent_unhappy = (
                self.num_unhappy / len(agents)
            ) * 100

    def move_unhappy_turtles(self):
        """
        Equivalente ao move-unhappy-turtles do NetLogo.
        """

        unhappy_agents = [
            agent for agent in list(self.agents)
            if not agent.happy
        ]

        self.random.shuffle(unhappy_agents)

        for agent in unhappy_agents:
            agent.step()

    def step(self):
        """
        Equivalente ao go do NetLogo.
        """

        if all(agent.happy for agent in list(self.agents)):
            self.running = False
            return

        self.move_unhappy_turtles()
        self.update_turtles()
        self.update_globals()

        self.tick += 1
        self.datacollector.collect(self)


## 5. Funções de visualização


In [ ]:
def plot_grid(model, title=None):
    """
    Visualiza o estado espacial da segregação.
    """

    fig, ax = plt.subplots(figsize=(8, 8))

    for agent in model.agents:
        x, y = agent.pos
        ax.scatter(
            x,
            y,
            c=agent.color,
            s=22,
            edgecolors="black",
            linewidths=0.2
        )

    ax.set_xlim(-1, model.width)
    ax.set_ylim(-1, model.height)
    ax.set_aspect("equal")

    ax.set_xlabel("x")
    ax.set_ylabel("y")

    if title is None:
        title = (
            f"Segregation Simple Extension 3 - tick {model.tick}\n"
            f"Similaridade média: {model.percent_similar:.2f}% | "
            f"Infelizes: {model.percent_unhappy:.2f}%"
        )

    ax.set_title(title)
    plt.show()


def plot_metrics(model):
    """
    Plota os monitores equivalentes ao NetLogo:
    - percent-similar
    - percent-unhappy
    """

    df = model.datacollector.get_model_vars_dataframe()

    plt.figure(figsize=(10, 5))
    plt.plot(df["tick"], df["percent_similar"], label="Percent similar")
    plt.plot(df["tick"], df["percent_unhappy"], label="Percent unhappy")
    plt.xlabel("Tick")
    plt.ylabel("Percentual")
    plt.title("Métricas globais da simulação")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    return df


## 6. Função para rodar a simulação


In [ ]:
def run_simulation(
    width=50,
    height=50,
    number=2000,
    number_of_ethnicities=5,
    percent_similar_wanted=30,
    percent_different_wanted=5,
    move_radius=10,
    max_steps=500,
    seed=42,
    show_final_grid=True,
    show_metrics=True
):
    """
    Executa a simulação completa.
    """

    model = SegregationModel(
        width=width,
        height=height,
        number=number,
        number_of_ethnicities=number_of_ethnicities,
        percent_similar_wanted=percent_similar_wanted,
        percent_different_wanted=percent_different_wanted,
        move_radius=move_radius,
        seed=seed
    )

    for _ in range(max_steps):
        if not model.running:
            break
        model.step()

    print("Simulação finalizada")
    print(f"Ticks executados: {model.tick}")
    print(f"Percent similar: {model.percent_similar:.2f}%")
    print(f"Percent unhappy: {model.percent_unhappy:.2f}%")
    print(f"Agentes felizes: {model.num_happy}")
    print(f"Agentes infelizes: {model.num_unhappy}")

    if show_final_grid:
        plot_grid(model)

    df_metrics = None

    if show_metrics:
        df_metrics = plot_metrics(model)

    return model, df_metrics


## 7. Executar cenário base

Este cenário usa valores próximos ao modelo original:

- grade 50 x 50;
- 2000 agentes;
- 5 etnias;
- similaridade desejada máxima de 30%;
- diferença desejada de 5%;
- raio de movimento 10.


In [ ]:
model, df_metrics = run_simulation(
    width=50,
    height=50,
    number=2000,
    number_of_ethnicities=5,
    percent_similar_wanted=30,
    percent_different_wanted=5,
    move_radius=10,
    max_steps=500,
    seed=42
)


## 8. Ver as métricas coletadas


In [ ]:
df_metrics.head()


In [ ]:
df_metrics.tail()


## 9. Exportar métricas para CSV


In [ ]:
df_metrics.to_csv("segregation_simple_extension3_metricas.csv", index=False)
print("Arquivo salvo: segregation_simple_extension3_metricas.csv")


## 10. Rodar cenários diferentes

Use esta seção para testar os efeitos dos parâmetros.


In [ ]:
cenarios = [
    {
        "nome": "baixo_similar_baixo_diferente",
        "percent_similar_wanted": 20,
        "percent_different_wanted": 5,
    },
    {
        "nome": "medio_similar_baixo_diferente",
        "percent_similar_wanted": 40,
        "percent_different_wanted": 5,
    },
    {
        "nome": "medio_similar_alto_diferente",
        "percent_similar_wanted": 40,
        "percent_different_wanted": 20,
    },
]

resultados = []

for cenario in cenarios:
    print("\nRodando cenário:", cenario["nome"])

    m, df = run_simulation(
        width=50,
        height=50,
        number=2000,
        number_of_ethnicities=5,
        percent_similar_wanted=cenario["percent_similar_wanted"],
        percent_different_wanted=cenario["percent_different_wanted"],
        move_radius=10,
        max_steps=500,
        seed=42,
        show_final_grid=False,
        show_metrics=False
    )

    resultados.append({
        "cenario": cenario["nome"],
        "ticks": m.tick,
        "percent_similar_final": m.percent_similar,
        "percent_unhappy_final": m.percent_unhappy,
        "num_happy_final": m.num_happy,
        "num_unhappy_final": m.num_unhappy,
    })

resultado_cenarios = pd.DataFrame(resultados)
resultado_cenarios


## 11. Comparar cenários em gráfico


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(
    resultado_cenarios["cenario"],
    resultado_cenarios["percent_unhappy_final"]
)
plt.xticks(rotation=30, ha="right")
plt.ylabel("% de agentes infelizes no final")
plt.title("Comparação dos cenários")
plt.grid(axis="y", alpha=0.3)
plt.show()


## 12. Observações para análise

Algumas perguntas úteis para interpretar os resultados:

1. O aumento de `% similar wanted` aumenta a segregação espacial?
2. O aumento de `% different wanted` dificulta que todos os agentes fiquem felizes?
3. Existem combinações de parâmetros em que o sistema não converge?
4. O número de etnias altera o tamanho e a nitidez dos agrupamentos?
5. O percentual médio de vizinhos similares no final é maior do que o limiar individual médio dos agentes?
